In [1]:
import pandas as pd

users = pd.DataFrame([
    {"user_id": 1, "education": "BCA", "interests": "technology coding"},
    {"user_id": 2, "education": "Commerce", "interests": "finance business"},
    {"user_id": 3, "education": "Arts", "interests": "design fashion"}
])

print(users)

   user_id education          interests
0        1       BCA  technology coding
1        2  Commerce   finance business
2        3      Arts     design fashion


In [5]:

products = pd.DataFrame([
    {
        "product_id": 101,
        "product_name": "Gaming Laptop",
        "category": "electronics",
        "tags": "technology coding programming student"
    },
    
    {
        "product_id": 102,
        "product_name": "Finance Book",
        "category": "books",
        "tags": "finance business commerce investment"
    },
    
    {
        "product_id": 103,
        "product_name": "Graphic Tablet",
        "category": "electronics",
        "tags": "design art creativity drawing"
    },
    
    {
        "product_id": 104,
        "product_name": "Programming Course",
        "category": "education",
        "tags": "coding technology software development"
    }
])

print(products)

   product_id        product_name     category  \
0         101       Gaming Laptop  electronics   
1         102        Finance Book        books   
2         103      Graphic Tablet  electronics   
3         104  Programming Course    education   

                                     tags  
0   technology coding programming student  
1    finance business commerce investment  
2           design art creativity drawing  
3  coding technology software development  


In [6]:
users["profile"] = users["education"] + " " + users["interests"]

print(users[["user_id", "profile"]])

   user_id                    profile
0        1      BCA technology coding
1        2  Commerce finance business
2        3        Arts design fashion


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
vectorizer = TfidfVectorizer()

product_vectors = vectorizer.fit_transform(products["tags"])

print(product_vectors)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16 stored elements and shape (4, 14)>
  Coords	Values
  (0, 13)	0.43779123108611473
  (0, 2)	0.43779123108611473
  (0, 10)	0.5552826649411127
  (0, 12)	0.5552826649411127
  (1, 8)	0.5
  (1, 1)	0.5
  (1, 3)	0.5
  (1, 9)	0.5
  (2, 5)	0.5
  (2, 0)	0.5
  (2, 4)	0.5
  (2, 7)	0.5
  (3, 13)	0.43779123108611473
  (3, 2)	0.43779123108611473
  (3, 11)	0.5552826649411127
  (3, 6)	0.5552826649411127


In [9]:
user_vectors = vectorizer.transform(users["profile"])

print(user_vectors)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (3, 14)>
  Coords	Values
  (0, 2)	0.7071067811865475
  (0, 13)	0.7071067811865475
  (1, 1)	0.5773502691896257
  (1, 3)	0.5773502691896257
  (1, 8)	0.5773502691896257
  (2, 5)	1.0


In [10]:
similarity = cosine_similarity(user_vectors, product_vectors)

print(similarity)

[[0.6191303 0.        0.        0.6191303]
 [0.        0.8660254 0.        0.       ]
 [0.        0.        0.5       0.       ]]


In [11]:
for i, user in users.iterrows():
    
    scores = similarity[i]
    
    best_product_index = scores.argmax()
    
    recommended_product = products.iloc[best_product_index]["product_name"]
    
    print(f"User {user['user_id']} should be recommended: {recommended_product}")

User 1 should be recommended: Gaming Laptop
User 2 should be recommended: Finance Book
User 3 should be recommended: Graphic Tablet


In [12]:
for i, user in users.iterrows():
    
    scores = similarity[i]
    
    sorted_indices = scores.argsort()[::-1]
    
    top_products = sorted_indices[:3]
    
    print(f"\nRecommendations for User {user['user_id']}:")
    
    for index in top_products:
        
        product_name = products.iloc[index]["product_name"]
        
        score = scores[index]
        
        print(f"{product_name}  | Similarity Score: {score:.2f}")


Recommendations for User 1:
Programming Course  | Similarity Score: 0.62
Gaming Laptop  | Similarity Score: 0.62
Graphic Tablet  | Similarity Score: 0.00

Recommendations for User 2:
Finance Book  | Similarity Score: 0.87
Programming Course  | Similarity Score: 0.00
Graphic Tablet  | Similarity Score: 0.00

Recommendations for User 3:
Graphic Tablet  | Similarity Score: 0.50
Programming Course  | Similarity Score: 0.00
Finance Book  | Similarity Score: 0.00


In [13]:
recommendations = []

for i, user in users.iterrows():
    
    scores = similarity[i]
    
    sorted_indices = scores.argsort()[::-1]
    
    top_products = sorted_indices[:3]
    
    for index in top_products:
        
        recommendations.append({
            "user_id": user["user_id"],
            "education": user["education"],
            "recommended_product": products.iloc[index]["product_name"],
            "score": round(scores[index], 2)
        })

recommendation_df = pd.DataFrame(recommendations)

print(recommendation_df)

   user_id education recommended_product  score
0        1       BCA  Programming Course   0.62
1        1       BCA       Gaming Laptop   0.62
2        1       BCA      Graphic Tablet   0.00
3        2  Commerce        Finance Book   0.87
4        2  Commerce  Programming Course   0.00
5        2  Commerce      Graphic Tablet   0.00
6        3      Arts      Graphic Tablet   0.50
7        3      Arts  Programming Course   0.00
8        3      Arts        Finance Book   0.00
